In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")

# llm = LLM(model="groq/openai/gpt-oss-20b", base_url="https://api.groq.com/openai/v1")

In [ ]:
# Setting up common configfor tool

_tool_config = dict(
    llm=dict(
        provider="ollama", # or google, openai, anthropic, llama2, ...
        config=dict(
            model="llama3.2:1b-instruct-q8_0",
            # temperature=0.5,
            # top_p=1,
            # stream=true,
        ),
    ),
    embedder=dict(
        provider="ollama", # or openai, ollama, ...
        config=dict(
            model_name="all-minilm",
            task_type="RETRIEVAL_DOCUMENT",
            # title="Embeddings",
        ),
    ),
)

#
_rag_tool_config = {
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "vectordb": {
        "provider": "chromadb",
        "config": {
            "persist_directory":"agentic-ai/chromadb",
            "allow_reset":"true",
            "is_persistent":"true",
            #"settings": Settings(persist_directory="agentic-ai/chromadb", allow_reset=True, is_persistent=True)
        }
    },
}
   

In [ ]:
from crewai_tools import CodeDocsSearchTool

@tool
def codedoc_search_tool(doc_path: str):
    """
    read api documentation and return content.
    """
    # by providing its URL:
    codedoc_search_tool = CodeDocsSearchTool(
        docs_url='https://docs.oracle.com/javase/8/docs/api/java/net/URL.html',
        config=_tool_config
    )

    #
    return codedoc_search_tool.run()

In [ ]:
import os
from crewai_tools import GithubSearchTool

#@tool
def github_search_tool(github_repo: str):
	"""
    read git hub repo and return repo, code, issues and pr.
    """
    # Initialize the tool for semantic searches within a specific GitHub repository
	_repo = 'https://github.com/brijeshdhaker'
	if github_repo :
		_repo = github_repo
	
	_git_hub_token = os.getenv("GIT_HUB_TOKEN")

	github_search_tool = GithubSearchTool(
		config=_rag_tool_config,
		#github_repo=_repo,
		gh_token=_git_hub_token,
		# Options: code, repo, pr, issue
		content_types=['repo'] 
	)

	return github_search_tool.run("docker-hadoop-cluster")

g_result = github_search_tool(github_repo='docker-hadoop-cluster')
g_result

In [ ]:
from crewai_tools import DOCXSearchTool
# Instantiate Web Search Tool

#@tool
def docx_search_tool(query: str, doc_path :str):
    """
    read workd document and return document content.
    """
	# Initialize the tool for semantic searches within a specific GitHub repository
    _doc_path = '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx'
    if doc_path :
        _doc_path = doc_path

    # Initialize the tool to search within any DOCX file's content
    docx_tool = DOCXSearchTool(docx=_doc_path, config=_rag_tool_config)
    #
    return docx_tool.run(query)


#
docx_results = docx_search_tool(query="Cloudera", doc_path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx")
docx_results

In [ ]:
from crewai.tools import tool
from crewai_tools import WebsiteSearchTool

# Instantiate Web Search Tool
@tool
def web_search_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )
    #
    return search_tool.run(query)


In [ ]:
from crewai_tools import ScrapeWebsiteTool


# Instantiate Web Search Tool
@tool
def scrap_website_tool(query: str):
    """
    scrap the content of the specified website.
    """
    # Initialize the tool with the website URL, 
    # so the agent can only scrap the content of the specified website
    scrap_website_tool = ScrapeWebsiteTool(
        website_url='https://example.com',
        config=_tool_config
    )
    
    # Extract the text from the site
    #text = scrap_website_tool.run()
    #print(text)

    #
    return scrap_website_tool.run(query)

In [ ]:
# 1. Initialize the tool
from crewai_tools import PDFSearchTool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
@tool
def pdf_search_tool(query : str):
        """_summary
            Searches the pdf documents and returns results.
        """
        pdf_tool = PDFSearchTool(
            pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
            config={
                "embedding_model": {
                    "provider": "openai",
                    "config": {
                        "model": "nomic-embed-text",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "embedder": {
                    "provider": "openai",
                    "config": {
                        "model": "all-minilm",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        "persist_directory":"agentic-ai/chromadb",
                        "allow_reset":"true",
                        "is_persistent":"true",
                    }
                },
            }
        )

        return pdf_tool.run(query)

#
results = pdf_search_tool.run("Cloudera")
results

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(data)
documents

##### Embaddings

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager
from com.example.agentic.splitter.SplitManager import SplitManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

#### Formatters

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key images or design about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Source image with title and url for each findings",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [ ]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [23]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool


@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "agentic-ai/crewai/config/agents.yaml"
    tasks_config = "agentic-ai/crewai/config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True,
            tools=[SerperDevTool()],
            llm=llm
        )
    
    @agent
    def image_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['image_researcher'],
            verbose=True,
            tools=[],
            allow_delegation=False,
            llm=llm
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True,
            llm=llm
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True,
            llm=llm
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'],
            output_json=ResearchOutput
        )
    
    @task
    def find_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['find_images_task'],
            output_json=ResearchImageOutput
        )
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )
		
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [24]:
from datetime import datetime

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': datetime.now().strftime('%Y-%m-%d')
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

In [ ]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 26a72cda-c9b0-4ea6-879e-e914b71e48ee                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: 37d9ba4c-a899-440c-be48-016431f33ed5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find any interesting and relevant    │
│  information about Microservice Design.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='Microservices are modular, distributed   │
│  systems that consist of multiple smaller services, each with its own logic and database.', relevance='The      │
│  main advantage of microservices is increased scalability and fault tolerance as individual service failures    │
│  can be contained within the service itself rather than affecting the entire system.',                          │
│  sources=[{'additionalProperties': '{}', 'type': 'string', 'properties': '{', 'name': 'search_query)',          │
│  'description': 'Not specified in the format', 'required': ':[{', 'title': 'Search Query'}, {'name': '{}',      │
│  'title': '{}', 'type': 'null'}])] summary='Microservices are increasing in popularity as they offer better     │
│  scalability, faster deployment, and increased flexibility.'                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Task: Format the executive report into a beautiful markdown document without '```'. Follow these guidelines:   │
│  1. Use proper markdown headers (#, ##, ###) 2. Include emojis from the section_emoji field 3. Format key       │
│  findings and insights as bullet points 4. Add proper spacing and sections breaks 5. Make recommendations       │
│  stand out using blockquotes 6. Ensure the date is properly formatted 7. Add a table of contents at the         │
│  beginning 8. Use horizontal rules (---) to separate major sections 9. Add inline citations for each major      │
│  claim using (Author, Year) format 10. Include a Sources section at the end with numbered references 11. Each   │
│  source should include title, publisher/author, and URL 12. Link inline citations to the corresponding entry    │
│  in the Sources section                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # 🔍 Introduction to Microservice Design Report                                                                │
│  ## 📆 Executive Summary                                                                                        │
│  Executive summary:                                                                                             │
│  This report provides an overview of microservice design principles, their benefits, and the current state of   │
│  industry trends. The report is designed to provide a comprehensive understanding of microservices in modern    │
│  software development.                                                                                          │
│                                                                                                                 │
│  ### Microservices are increasing in popularity as they offer better scalability, faster deployment, and        │
│  increased flexibility.                                                                                         │
│  Benefits of microservices design:                                                                              │
│  * Increased flexibility in deployment and management                                                           │
│  * Efficient use of resources including data, database schema, codebase organization, communication between     │
│  services                                                                                                       │
│  * Improved overall system agility                                                                              │
│                                                                                                                 │
│  Risk associated with microservices design:                                                                     │
│  Complex configuration updates required for all services on each deployment which can be challenging.           │
│                                                                                                                 │
│  Recommendations:                                                                                               │
│  ### Benefits of Microservices Design 💡                                                                        │
│  Increased flexibility in deployment and management, efficient use of resources including data, database        │
│  schema, codebase organization, communication between services                                                  │
│  The ability to scale individual services independently while maintaining overall system agility                │
│                                                                                                                 │
│  ### Risks Associated with Microservices Design 🚨                                                              │
│  Risk of complex communication between services                                                                 │
│  Complex configuration updates required for all services on each deployment which can be challenging            │
│                                                                                                                 │
│  ### Key Findings and Insights 🔍                                                                               │
│  ## **Key Findings**                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: formatting_task                                                                                          │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 8c072165-4b7a-4ebc-8405-a80c225ea2c3                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/8c072165-4b7a-4eb │
│ c-8405-a80c225ea2c3?access_code=TRACE-52b0bcb50a                             │
│ 🔑 Access Code: TRACE-52b0bcb50a                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 26a72cda-c9b0-4ea6-879e-e914b71e48ee                                                                       │
│  Final Output: # 🔍 Introduction to Microservice Design Report                                                  │
│  ## 📆 Executive Summary                                                                                        │
│  Executive summary:                                                                                             │
│  This report provides an overview of microservice design principles, their benefits, and the current state of   │
│  industry trends. The report is designed to provide a comprehensive understanding of microservices in modern    │
│  software development.                                                                                          │
│                                                                                                                 │
│  ### Microservices are increasing in popularity as they offer better scalability, faster deployment, and        │
│  increased flexibility.                                                                                         │
│  Benefits of microservices design:                                                                              │
│  * Increased flexibility in deployment and management                                                           │
│  * Efficient use of resources including data, database schema, codebase organization, communication between     │
│  services                                                                                                       │
│  * Improved overall system agility                                                                              │
│                                                                                                                 │
│  Risk associated with microservices design:                                                                     │
│  Complex configuration updates required for all services on each deployment which can be challenging.           │
│                                                                                                                 │
│  Recommendations:                                                                                               │
│  ### Benefits of Microservices Design 💡                                                                        │
│  Increased flexibility in deployment and management, efficient use of resources including data, database        │
│  schema, codebase organization, communication between services                                                  │
│  The ability to scale individual services independently while maintaining overall system agility                │
│                                                                                                                 │
│  ### Risks Associated with Microservices Design 🚨                                                              │
│  Risk of complex communication between services                                                                 │
│  Complex configuration updates required for all services on each deployment which can be challenging            │
│                                                                                                                 │
│  ### Key Findings and Insights 🔍                                                                               │
│  ## **Key Findings**                                       

####
####